# 🔍 AML Layering Detection Engine
**System:** Anti-Money Laundering Detection | Neo4j Aura Enterprise 5.27  
**Module Coverage:** Module 1 → Module 7

---

### Architecture
```
Module 1  : Seed Selection        (rank accounts by degree, fan-out biased)
Module 2  : Seed Batching
Module 2A : Fan-out Detector      (1 source → N receivers)
Module 2B : Fan-in Detector       (N sources → 1 receiver)
Module 3  : Rapid Movement Detector (pass-through / quick forwarding)
Module 4  : Merge & Remove Duplicates
Module 5  : Feature Engineering    (laundering_ratio intentionally excluded)
Module 6  : Risk Scoring
Module 7  : Validation & Evaluation
```

**Note on Module 5:** `laundering_ratio` is deliberately excluded from the engineered
feature set. The `any_laundering` ground-truth label is still carried through for
evaluation in Module 7, and used as a low-weight signal in Module 6 scoring —
but no *ratio*-based laundering feature is engineered.

---
## ⚙️ Setup
Mount Google Drive and install required packages.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install neo4j pandas scikit-learn --quiet

---
## 🔌 Connect to Neo4j
Establish the driver connection using credentials stored in Colab Secrets.

In [1]:
!pip install neo4j pandas scikit-learn --quiet

In [3]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, timezone
from google.colab import userdata
from neo4j import GraphDatabase

URI      = userdata.get('NEO4J_URI').strip()
USERNAME = userdata.get('NEO4J_USERNAME').strip()
PASSWORD = userdata.get('NEO4J_PASSWORD').strip()

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print('✓ Connected to Neo4j AuraDB')

✓ Connected to Neo4j AuraDB


---
## MODULE 1 — Seed Selection
---

### MODULE 1 — Cell 1 : Compute Account Degree Statistics
Layering seeds need to be ranked differently from cycle seeds — fan-out and fan-in
patterns are driven by **degree** (how many distinct counterparties an account has),
not just transaction amount. This query computes in-degree, out-degree, and total
amount for every account in one pass.

In [4]:
print('Computing account degree statistics...')

ACCOUNT_DEGREE_QUERY = """
MATCH (a:Account)

OPTIONAL MATCH (a)-[out:TRANSACTION]->(receiver:Account)
WITH
    a,
    count(DISTINCT receiver)            AS out_degree,
    count(out)                          AS out_tx_count,
    coalesce(sum(out.amount_paid), 0)   AS total_out_amount

OPTIONAL MATCH (sender:Account)-[inc:TRANSACTION]->(a)
WITH
    a,
    out_degree,
    out_tx_count,
    total_out_amount,
    count(DISTINCT sender)              AS in_degree,
    count(inc)                          AS in_tx_count,
    coalesce(sum(inc.amount_paid), 0)   AS total_in_amount

RETURN
    a.account_id AS account_id,
    out_degree,
    in_degree,
    out_tx_count,
    in_tx_count,
    total_out_amount,
    total_in_amount,
    (total_out_amount + total_in_amount) AS total_transaction_amount
"""

with driver.session() as session:
    account_stats = session.run(ACCOUNT_DEGREE_QUERY).data()

account_df = pd.DataFrame(account_stats)
print(f'Accounts analysed : {len(account_df)}')
account_df.head()

Computing account degree statistics...
Accounts analysed : 43878


,account_id,out_degree,in_degree,out_tx_count,in_tx_count,total_out_amount,total_in_amount,total_transaction_amount
0,70_100428660,14015,523,132870,850,4.153677e+10,381704.99,4.153715e+10
1,1467_8013C4030,3,1,8,3,7.150404e+04,10300.00,8.180404e+04
2,119_811C597B0,5,6,26,39,4.017714e+07,66209821.35,1.063870e+08
3,70_100428858,972,37,7553,40,2.276012e+09,19935.02,2.276032e+09
4,21174_800737690,13,1,20,8,5.924033e+06,55784.11,5.979818e+06


### MODULE 1 — Cell 2 : Compute Layering Seed Score & Select Top Accounts
The seed score is weighted toward **degree** rather than amount, since layering is
fundamentally about the *number of counterparties* an account touches. We keep a
smaller amount weight because high-value fan-out/fan-in is still more suspicious
than low-value fan-out/fan-in of the same shape.

In [5]:
def normalize(series):
    if series.max() == series.min():
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - series.min()) / (series.max() - series.min())


# Degree signal = the stronger of out_degree or in_degree (captures either
# a fan-out hub or a fan-in hub with a single score)
account_df['max_degree'] = account_df[['out_degree', 'in_degree']].max(axis=1)

account_df['score_degree'] = normalize(account_df['max_degree'])
account_df['score_amount'] = normalize(account_df['total_transaction_amount'])

# Layering seed score: degree-biased (0.65) vs cycle's amount-biased scoring
account_df['layering_seed_score'] = (
      0.65 * account_df['score_degree']
    + 0.35 * account_df['score_amount']
).round(4)

account_df = account_df.sort_values(
    by=['layering_seed_score', 'account_id'],
    ascending=[False, True]
).reset_index(drop=True)

# Select top 10% as candidate seeds
num_seeds = int(len(account_df) * 0.10)
seed_accounts = account_df.head(num_seeds).copy()
seed_ids = seed_accounts['account_id'].tolist()

seed_accounts.to_csv('/content/layering_seed_accounts.csv', index=False)

print('Layering Seed Selection Completed')
print(f'Total Accounts          : {len(account_df)}')
print(f'Seed Accounts Selected  : {len(seed_ids)}')
seed_accounts.head(10)

Layering Seed Selection Completed
Total Accounts          : 43878
Seed Accounts Selected  : 4387


,account_id,out_degree,in_degree,out_tx_count,in_tx_count,total_out_amount,total_in_amount,total_transaction_amount,max_degree,score_degree,score_amount,layering_seed_score
0,70_100428660,14015,523,132870,850,4.153677e+10,3.817050e+05,4.153715e+10,14015,1.000000,0.159611,0.7059
1,70_1004286A8,8585,315,80389,512,2.029276e+10,1.887023e+05,2.029294e+10,8585,0.612530,0.077978,0.4254
2,70_100428780,1311,52,12508,82,2.602359e+11,3.345084e+06,2.602392e+11,1311,0.093478,1.000000,0.4108
3,70_100428738,1106,47,10747,78,1.711019e+11,5.298328e+06,1.711072e+11,1106,0.078850,0.657500,0.2814
4,70_1004287C8,1085,46,9520,62,6.513550e+10,2.679163e+06,6.513818e+10,1085,0.077351,0.250301,0.1379
5,13029_805C2B8A0,0,1,0,1,0.000000e+00,6.641449e+10,6.641449e+10,1,0.000000,0.255206,0.0893
6,14381_805C2AFB0,1,1,1,1,6.641449e+10,1.643140e+03,6.641450e+10,1,0.000000,0.255206,0.0893
7,70_100428978,1688,73,15978,124,6.315977e+09,3.601269e+04,6.316013e+09,1688,0.120380,0.024270,0.0867
8,70_1004286F0,1445,43,11943,53,1.440433e+10,1.242279e+05,1.440445e+10,1445,0.103040,0.055351,0.0863
9,70_1004288E8,701,22,6291,38,3.784097e+10,2.853579e+05,3.784126e+10,701,0.049950,0.145409,0.0834


### MODULE 1 — Cell 3 : Filter Eligible Seeds
A fan-out seed needs at least `MIN_FANOUT` distinct outgoing counterparties; a
fan-in seed needs at least `MIN_FANIN` distinct incoming counterparties. An
account only needs to qualify for **one** of these to be eligible — it will be
tried against whichever detector(s) it qualifies for.

In [6]:
MIN_FANOUT = 3   # minimum distinct receivers to be a fan-out candidate
MIN_FANIN  = 3   # minimum distinct senders to be a fan-in candidate

eligible_mask = (
    (seed_accounts['out_degree'] >= MIN_FANOUT) |
    (seed_accounts['in_degree']  >= MIN_FANIN)
)

eligible_seed_df = seed_accounts[eligible_mask].copy()
eligible_seed_ids = eligible_seed_df['account_id'].tolist()

# Split eligible seeds by which detector(s) they qualify for
fanout_seed_ids = eligible_seed_df.loc[
    eligible_seed_df['out_degree'] >= MIN_FANOUT, 'account_id'
].tolist()

fanin_seed_ids = eligible_seed_df.loc[
    eligible_seed_df['in_degree'] >= MIN_FANIN, 'account_id'
].tolist()

# Rapid-movement candidates need both an incoming and outgoing side
rapid_seed_ids = eligible_seed_df.loc[
    (eligible_seed_df['in_degree'] >= 1) & (eligible_seed_df['out_degree'] >= 1),
    'account_id'
].tolist()

print(f'Total seed accounts        : {len(seed_ids)}')
print(f'Eligible seed accounts     : {len(eligible_seed_ids)}')
print(f'  → Fan-out candidates     : {len(fanout_seed_ids)}')
print(f'  → Fan-in candidates      : {len(fanin_seed_ids)}')
print(f'  → Rapid-movement cands.  : {len(rapid_seed_ids)}')

Total seed accounts        : 4387
Eligible seed accounts     : 563
  → Fan-out candidates     : 349
  → Fan-in candidates      : 399
  → Rapid-movement cands.  : 497


---
## MODULE 2 — Seed Batching
---

### MODULE 2 — Cell 1 : Batch Each Candidate List
Each of the three seed lists (fan-out, fan-in, rapid) is split into batches of
200 to keep individual Cypher calls fast and within Aura's query limits.

In [7]:
BATCH_SIZE = 200

def make_batches(id_list, batch_size=BATCH_SIZE):
    return [id_list[i:i + batch_size] for i in range(0, len(id_list), batch_size)]

fanout_batches = make_batches(fanout_seed_ids)
fanin_batches  = make_batches(fanin_seed_ids)
rapid_batches  = make_batches(rapid_seed_ids)

print('=' * 60)
print('Seed Batch Summary')
print('=' * 60)
print(f'Fan-out batches : {len(fanout_batches):>4}  ({len(fanout_seed_ids)} seeds)')
print(f'Fan-in  batches : {len(fanin_batches):>4}  ({len(fanin_seed_ids)} seeds)')
print(f'Rapid   batches : {len(rapid_batches):>4}  ({len(rapid_seed_ids)} seeds)')
print('=' * 60)

Seed Batch Summary
Fan-out batches :    2  (349 seeds)
Fan-in  batches :    2  (399 seeds)
Rapid   batches :    3  (497 seeds)


---
## MODULE 2A / 2B / 3 — Detection Queries
---

### Cell 1 : Fan-out Detector Query
For each seed acting as a hub, collect every outgoing transaction and its receiver.
Time-window and minimum-fan-out filtering happens in Python after retrieval —
this keeps the Cypher query itself simple and fast, and avoids duration math
inside Cypher which is harder to reason about on Aura.

In [8]:
FAN_OUT_QUERY = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})-[t:TRANSACTION]->(b:Account)

WITH a, collect({
    receiver   : b.account_id,
    amount     : t.amount_paid,
    timestamp  : t.timestamp,
    laundering : t.is_laundering
}) AS out_txns

RETURN
    a.account_id AS hub_account,
    out_txns
"""

print('Fan-out detector query defined ✓')

Fan-out detector query defined ✓


### Cell 2 : Fan-in Detector Query
Mirror of the fan-out query — for each seed acting as a receiving hub, collect
every incoming transaction and its sender.

In [9]:
FAN_IN_QUERY = """
UNWIND $seed_ids AS origin_id

MATCH (b:Account)-[t:TRANSACTION]->(a:Account {account_id: origin_id})

WITH a, collect({
    sender     : b.account_id,
    amount     : t.amount_paid,
    timestamp  : t.timestamp,
    laundering : t.is_laundering
}) AS in_txns

RETURN
    a.account_id AS hub_account,
    in_txns
"""

print('Fan-in detector query defined ✓')

Fan-in detector query defined ✓


### Cell 3 : Rapid Movement (Pass-Through) Detector Query
Detects A → hub → C patterns where funds are forwarded onward, excluding
the trivial case where C equals A (that would just be a 2-hop cycle, already
covered by the cycle detection module). Both incoming and outgoing transaction
pairs for the hub are returned so Python can compute the forwarding delay.

In [10]:
RAPID_MOVEMENT_QUERY = """
UNWIND $seed_ids AS origin_id

MATCH (source:Account)-[t_in:TRANSACTION]->(hub:Account {account_id: origin_id})
     -[t_out:TRANSACTION]->(dest:Account)

WHERE source.account_id <> dest.account_id
  AND source.account_id <> hub.account_id
  AND dest.account_id   <> hub.account_id

  // t.timestamp is a native DATE_TIME — comparisons and duration.between
  // work directly, no conversion needed
  AND t_out.timestamp >= t_in.timestamp
  AND duration.between(t_in.timestamp, t_out.timestamp).seconds <= $window_seconds

RETURN
    hub.account_id     AS hub_account,
    source.account_id  AS source_account,
    dest.account_id    AS dest_account,
    t_in.amount_paid    AS amount_in,
    t_out.amount_paid   AS amount_out,
    t_in.timestamp      AS timestamp_in,
    t_out.timestamp     AS timestamp_out,
    t_in.is_laundering   AS laundering_in,
    t_out.is_laundering  AS laundering_out

LIMIT 5000
"""

print('Rapid movement query confirmed — DATE_TIME native type, no conversion needed ✓')

Rapid movement query confirmed — DATE_TIME native type, no conversion needed ✓


### Cell 4 : Single-Batch Executor Helper
Shared executor for all three detectors. Isolates error handling so one slow
or timed-out batch never crashes the full sweep.

In [21]:
def run_query_on_batch(session, query, batch, label, **extra_params):
    """Execute one Cypher query for one seed batch, with isolated error handling."""
    try:
        result = session.run(query, seed_ids=batch, **extra_params)
        return result.data()
    except Exception as e:
        print(f'  [WARN] {label} | batch failed → {e}')
        return []

print('Executor helper defined ✓')

Executor helper defined ✓


### Cell 5 : Run Fan-out & Fan-in Sweeps
Runs both detectors across their respective batch lists and prints progress.
These two run independently since they use different seed pools.

In [22]:
raw_fanout = []
raw_fanin  = []

print('=' * 65)
print('Running Fan-out Sweep')
print('=' * 65)

start = time.time()
with driver.session() as session:
    for i, batch in enumerate(fanout_batches):
        raw_fanout.extend(run_query_on_batch(session, FAN_OUT_QUERY, batch, 'fan-out'))
        if (i + 1) % 10 == 0 or (i + 1) == len(fanout_batches):
            print(f'  Batch {i+1:>4}/{len(fanout_batches)} | rows so far: {len(raw_fanout)}')
print(f'Fan-out sweep done in {time.time()-start:.1f}s — {len(raw_fanout)} hub records')

print()
print('=' * 65)
print('Running Fan-in Sweep')
print('=' * 65)

start = time.time()
with driver.session() as session:
    for i, batch in enumerate(fanin_batches):
        raw_fanin.extend(run_query_on_batch(session, FAN_IN_QUERY, batch, 'fan-in'))
        if (i + 1) % 10 == 0 or (i + 1) == len(fanin_batches):
            print(f'  Batch {i+1:>4}/{len(fanin_batches)} | rows so far: {len(raw_fanin)}')
print(f'Fan-in sweep done in {time.time()-start:.1f}s — {len(raw_fanin)} hub records')

Running Fan-out Sweep
  Batch    2/2 | rows so far: 349
Fan-out sweep done in 43.9s — 349 hub records

Running Fan-in Sweep
  Batch    2/2 | rows so far: 399
Fan-in sweep done in 1.2s — 399 hub records


### Cell 6 : Run Rapid Movement Sweep

> Add blockquote


Runs the pass-through detector across its own batch list.

In [23]:
RAPID_WINDOW_SECONDS = 3600
raw_rapid = []

print('=' * 65)
print('Running Rapid Movement Sweep')
print('=' * 65)

# Reduce batch size for this query specifically — high-degree hubs
# make this the most expensive detector, so smaller batches finish
# each round-trip faster and let you see progress sooner.
RAPID_BATCH_SIZE = 50
rapid_batches_small = make_batches(rapid_seed_ids, batch_size=RAPID_BATCH_SIZE)

start = time.time()
with driver.session() as session:
    for i, batch in enumerate(rapid_batches_small):
        raw_rapid.extend(
            run_query_on_batch(
                session, RAPID_MOVEMENT_QUERY, batch, 'rapid',
                window_seconds=RAPID_WINDOW_SECONDS
            )
        )
        if (i + 1) % 5 == 0 or (i + 1) == len(rapid_batches_small):
            elapsed = time.time() - start
            print(f'  Batch {i+1:>4}/{len(rapid_batches_small)} | rows so far: {len(raw_rapid)} | {elapsed:.1f}s')

print(f'Rapid movement sweep done in {time.time()-start:.1f}s — {len(raw_rapid)} pass-through records')

Running Rapid Movement Sweep
  Batch    5/10 | rows so far: 5815 | 5.4s
  Batch   10/10 | rows so far: 6046 | 6.5s
Rapid movement sweep done in 6.5s — 6046 pass-through records


---
## MODULE 4 — Merge & Remove Duplicates
---

### Cell 1 : Expand Fan-out Results into Time-Windowed Groups
Each hub's raw transaction list is inspected in Python: timestamps are parsed,
and only groups of transactions falling within `FANOUT_WINDOW_SECONDS` of each
other, with at least `MIN_FANOUT` distinct receivers, are kept as true fan-out events.

In [24]:
FANOUT_WINDOW_SECONDS = 86400   # 24 hours — tune based on dataset time granularity


def to_epoch(ts):
    """Convert a Neo4j DateTime object or ISO string to a Unix timestamp."""
    if ts is None:
        return None
    if hasattr(ts, 'to_native'):
        return ts.to_native().replace(tzinfo=timezone.utc).timestamp()
    if isinstance(ts, str):
        return datetime.fromisoformat(ts).timestamp()
    return float(ts)


def extract_fanout_events(raw_records, window_seconds=FANOUT_WINDOW_SECONDS, min_fanout=MIN_FANOUT):
    """
    For each hub, sort its outgoing transactions by time and keep only the
    subset that falls within `window_seconds` of the first transaction AND
    has at least `min_fanout` distinct receivers. Returns one row per
    qualifying hub.
    """
    events = []

    for record in raw_records:
        hub = record['hub_account']
        txns = record.get('out_txns') or []

        # Attach parsed epoch time to each transaction, drop unparseable ones
        parsed = []
        for tx in txns:
            epoch = to_epoch(tx.get('timestamp'))
            if epoch is not None:
                parsed.append({**tx, 'epoch': epoch})

        if not parsed:
            continue

        parsed.sort(key=lambda x: x['epoch'])

        # Keep transactions within window_seconds of the first transaction
        window_start = parsed[0]['epoch']
        windowed = [tx for tx in parsed if tx['epoch'] - window_start <= window_seconds]

        distinct_receivers = {tx['receiver'] for tx in windowed}

        if len(distinct_receivers) >= min_fanout:
            events.append({
                'hub_account'   : hub,
                'pattern_type'  : 'fan_out',
                'counterparties': sorted(distinct_receivers),
                'fanout_count'  : len(distinct_receivers),
                'amounts'       : [tx['amount'] for tx in windowed],
                'timestamps'    : [tx['timestamp'] for tx in windowed],
                'is_laundering' : [tx['laundering'] for tx in windowed],
            })

    return events


fanout_events = extract_fanout_events(raw_fanout)
print(f'Fan-out events extracted : {len(fanout_events)}')

Fan-out events extracted : 234


### Cell 2 : Expand Fan-in Results into Time-Windowed Groups
Same logic as fan-out, applied to incoming transactions.

In [25]:
FANIN_WINDOW_SECONDS = 86400


def extract_fanin_events(raw_records, window_seconds=FANIN_WINDOW_SECONDS, min_fanin=MIN_FANIN):
    """Same windowing logic as fan-out, applied to incoming transactions."""
    events = []

    for record in raw_records:
        hub = record['hub_account']
        txns = record.get('in_txns') or []

        parsed = []
        for tx in txns:
            epoch = to_epoch(tx.get('timestamp'))
            if epoch is not None:
                parsed.append({**tx, 'epoch': epoch})

        if not parsed:
            continue

        parsed.sort(key=lambda x: x['epoch'])

        window_start = parsed[0]['epoch']
        windowed = [tx for tx in parsed if tx['epoch'] - window_start <= window_seconds]

        distinct_senders = {tx['sender'] for tx in windowed}

        if len(distinct_senders) >= min_fanin:
            events.append({
                'hub_account'   : hub,
                'pattern_type'  : 'fan_in',
                'counterparties': sorted(distinct_senders),
                'fanout_count'  : len(distinct_senders),   # reuse column name for uniformity
                'amounts'       : [tx['amount'] for tx in windowed],
                'timestamps'    : [tx['timestamp'] for tx in windowed],
                'is_laundering' : [tx['laundering'] for tx in windowed],
            })

    return events


fanin_events = extract_fanin_events(raw_fanin)
print(f'Fan-in events extracted : {len(fanin_events)}')

Fan-in events extracted : 168


### Cell 3 : Filter Rapid Movement Pairs by Forwarding Delay
Keeps only source → hub → dest triples where the outgoing transaction happened
within `RAPID_WINDOW_SECONDS` of the incoming one — the signature of a
pass-through / mule account.

In [26]:
RAPID_WINDOW_SECONDS = 3600   # 1 hour — tune based on dataset time granularity


def extract_rapid_events(raw_records, window_seconds=RAPID_WINDOW_SECONDS):
    """
    Keep only pass-through triples where funds left the hub within
    window_seconds of arriving — and only when the outgoing transaction
    happened AFTER the incoming one (proper forwarding, not coincidence).
    """
    events = []

    for record in raw_records:
        epoch_in  = to_epoch(record.get('timestamp_in'))
        epoch_out = to_epoch(record.get('timestamp_out'))

        if epoch_in is None or epoch_out is None:
            continue

        delay = epoch_out - epoch_in

        if 0 <= delay <= window_seconds:
            events.append({
                'hub_account'   : record['hub_account'],
                'pattern_type'  : 'rapid_movement',
                'counterparties': sorted({record['source_account'], record['dest_account']}),
                'fanout_count'  : 2,
                'amounts'       : [record['amount_in'], record['amount_out']],
                'timestamps'    : [record['timestamp_in'], record['timestamp_out']],
                'is_laundering' : [record['laundering_in'], record['laundering_out']],
                'forward_delay_seconds': delay,
            })

    return events


rapid_events = extract_rapid_events(raw_rapid)
print(f'Rapid movement events extracted : {len(rapid_events)}')

Rapid movement events extracted : 1442


### Cell 4 : Merge All Detector Outputs & Deduplicate
Combines fan-out, fan-in, and rapid-movement events into one DataFrame, then
deduplicates using a fingerprint of `(hub_account, pattern_type, sorted
counterparties)` — this prevents the same hub/counterparty combination from
appearing twice if two overlapping seed batches both surfaced it.

In [27]:
all_events = fanout_events + fanin_events + rapid_events

if all_events:
    layering_raw_df = pd.DataFrame(all_events)

    def make_fingerprint(row):
        cp_key = '|'.join(row['counterparties'])
        return f"{row['hub_account']}::{row['pattern_type']}::{cp_key}"

    layering_raw_df['fingerprint'] = layering_raw_df.apply(make_fingerprint, axis=1)

    rows_before = len(layering_raw_df)
    layering_raw_df = layering_raw_df.drop_duplicates(
        subset=['fingerprint'], keep='first'
    ).reset_index(drop=True)
    rows_after = len(layering_raw_df)

    print(f'Total events before dedup : {rows_before}')
    print(f'Total events after  dedup : {rows_after}')
    print(f'Duplicates removed        : {rows_before - rows_after}')
    print()
    print('Breakdown by pattern type:')
    print(layering_raw_df['pattern_type'].value_counts().to_string())

else:
    layering_raw_df = pd.DataFrame(columns=[
        'hub_account', 'pattern_type', 'counterparties', 'fanout_count',
        'amounts', 'timestamps', 'is_laundering', 'fingerprint'
    ])
    print('No layering events detected across any detector.')

layering_raw_df.head()

Total events before dedup : 1844
Total events after  dedup : 1227
Duplicates removed        : 617

Breakdown by pattern type:
pattern_type
rapid_movement    825
fan_out           234
fan_in            168


,hub_account,pattern_type,counterparties,fanout_count,amounts,timestamps,is_laundering,forward_delay_seconds,fingerprint
0,70_100428660,fan_out,"[10232_803E422C0, 10232_804AF10C0, 10232_80515...",10119,"[23317.0, 18722.79, 4815.41, 717.6, 792.92, 16...","[2022-09-01T00:01:00.000000000+00:00, 2022-09-...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_100428660::fan_out::10232_803E422C0|10232_8...
1,70_1004286A8,fan_out,"[1024_80069C8D0, 1024_8006CED50, 1024_80070EFF...",6117,"[834.69, 3673.87, 18741.89, 146614.28, 1962.5,...","[2022-09-01T00:12:00.000000000+00:00, 2022-09-...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_1004286A8::fan_out::1024_80069C8D0|1024_800...
2,70_100428780,fan_out,"[112064_8048204B0, 112064_804824390, 112064_80...",891,"[152.58, 1542.58, 91915.07, 336528.23, 108101....","[2022-09-01T03:07:00.000000000+00:00, 2022-09-...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_100428780::fan_out::112064_8048204B0|112064...
3,70_100428738,fan_out,"[10057_803AACE20, 10057_803AB8EB0, 10057_803B1...",783,"[108079.61, 38326.81, 6354428.3, 4441607.27, 2...","[2022-09-01T00:09:00.000000000+00:00, 2022-09-...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_100428738::fan_out::10057_803AACE20|10057_8...
4,70_1004287C8,fan_out,"[10_80609E620, 10_8060A4F90, 10_8060B6E20, 10_...",741,"[3874814.83, 2087266.56, 28999429.29, 957969.2...","[2022-09-01T01:47:00.000000000+00:00, 2022-09-...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_1004287C8::fan_out::10_80609E620|10_8060A4F...


---
## MODULE 5 — Feature Engineering
*(laundering_ratio intentionally excluded — see note below)*
---

### Cell 1 : Define Feature Engineering Function
Derives amount, temporal, and structural features from each layering event.
**`laundering_ratio` is not computed here** — only the binary `any_laundering`
flag is carried through, and only for use as ground truth in Module 7 and as a
single low-weight signal in Module 6. No ratio-based laundering feature is engineered.

In [28]:
def engineer_layering_features(row: pd.Series) -> pd.Series:
    """
    Compute derived features for a single layering event (fan-out, fan-in,
    or rapid movement). Feature groups:

      Amount features    : total, min, max, avg, std, range
      Structural features: fanout_count (already present), unique_counterparties
      Temporal features  : time_span_seconds, avg_tx_interval, min_tx_interval,
                            spread_speed (counterparties per hour)
      Label feature       : any_laundering  (binary ground-truth flag only —
                             NOT a ratio; used solely for evaluation / scoring)
    """
    amounts    = row.get('amounts')    or []
    timestamps = row.get('timestamps') or []
    labels     = row.get('is_laundering') or []

    # ── Amount features ─────────────────────────────────
    amounts_clean = [a for a in amounts if a is not None]
    total_amount  = sum(amounts_clean)
    min_amount    = min(amounts_clean) if amounts_clean else 0.0
    max_amount    = max(amounts_clean) if amounts_clean else 0.0
    avg_amount    = float(np.mean(amounts_clean)) if amounts_clean else 0.0
    amount_std    = float(np.std(amounts_clean))  if len(amounts_clean) > 1 else 0.0
    amount_range  = max_amount - min_amount

    # ── Label feature (binary only — no ratio) ─────────────────
    labels_clean   = [bool(l) for l in labels if l is not None]
    any_laundering = int(sum(labels_clean) > 0)

    # ── Structural features ────────────────────────────
    counterparties = row.get('counterparties') or []
    unique_counterparties = len(set(counterparties))

    # ── Temporal features ─────────────────────────────
    time_span_seconds = 0.0
    avg_tx_interval   = 0.0
    min_tx_interval   = 0.0
    spread_speed      = 0.0   # counterparties reached per hour

    try:
        epochs = sorted([to_epoch(t) for t in timestamps if t is not None])
        epochs = [e for e in epochs if e is not None]

        if len(epochs) >= 2:
            time_span_seconds = epochs[-1] - epochs[0]
            intervals         = [epochs[i+1] - epochs[i] for i in range(len(epochs)-1)]
            avg_tx_interval   = float(np.mean(intervals))
            min_tx_interval   = float(min(intervals))

            if time_span_seconds > 0:
                spread_speed = unique_counterparties / (time_span_seconds / 3600.0)

    except Exception:
        pass

    return pd.Series({
        'total_amount'          : total_amount,
        'min_amount'            : min_amount,
        'max_amount'            : max_amount,
        'avg_amount'            : avg_amount,
        'amount_std'            : amount_std,
        'amount_range'          : amount_range,
        'any_laundering'        : any_laundering,
        'unique_counterparties' : unique_counterparties,
        'time_span_seconds'     : time_span_seconds,
        'avg_tx_interval'       : avg_tx_interval,
        'min_tx_interval'       : min_tx_interval,
        'spread_speed'          : spread_speed,
    })


print('Feature engineering function defined ✓ (no laundering_ratio included)')

Feature engineering function defined ✓ (no laundering_ratio included)


### Cell 2 : Apply Feature Engineering
Applies the function row-wise and joins the resulting columns onto the base
DataFrame to produce `enriched_df`.

In [29]:
if not layering_raw_df.empty:

    print('Applying feature engineering...')

    feature_df = layering_raw_df.apply(engineer_layering_features, axis=1)

    enriched_df = pd.concat(
        [layering_raw_df.reset_index(drop=True), feature_df.reset_index(drop=True)],
        axis=1
    )

    print(f'Feature engineering complete — {len(enriched_df)} events enriched')
    print(f'Total columns : {len(enriched_df.columns)}')
    assert 'laundering_ratio' not in enriched_df.columns, 'laundering_ratio must not be present'
    print('Confirmed: laundering_ratio is NOT in the feature set ✓')

    enriched_df[[
        'hub_account', 'pattern_type', 'fanout_count',
        'total_amount', 'spread_speed', 'any_laundering'
    ]].head()

else:
    enriched_df = layering_raw_df.copy()
    print('No events to engineer features for.')

Applying feature engineering...
Feature engineering complete — 1227 events enriched
Total columns : 21
Confirmed: laundering_ratio is NOT in the feature set ✓


---
## MODULE 6 — Risk Scoring
---

### Cell 1 : Define Risk Scoring Functions
Weights follow the architecture: fan-out/structural signal (0.35), amount (0.25),
velocity (0.25), and the binary laundering label (0.15). Note the label
component uses `any_laundering` — a 0/1 flag — not a ratio.

In [30]:
def compute_layering_risk_score(row: pd.Series) -> float:
    """
    Composite risk score for one layering event. Range: 0.0 – 1.0

    Dimension         Weight  Rationale
    ────────────────────────────────────────────
    fanout_score       0.35   More counterparties = more layering structure
    amount_score       0.25   Higher total value raises suspicion
    velocity_score     0.25   Faster spread across counterparties = higher risk
    label_score         0.15   any_laundering ground-truth flag (binary, not ratio)
    """

    # ── Fan-out / structural score ─────────────────
    # Normalise fanout_count on a scale where 3 counterparties → 0, 20+ → 1
    fanout_count = float(row.get('fanout_count', 0))
    fanout_score = min(max(fanout_count - 3, 0) / 17.0, 1.0)

    # ── Amount score (log-normalised, capped at 1) ─────────
    total = float(row.get('total_amount', 0.0))
    amount_score = min(np.log10(total + 1) / 6.0, 1.0) if total > 0 else 0.0

    # ── Velocity score (faster spread = higher score) ────────
    # spread_speed is counterparties/hour; 0 → 0.0, 10+/hr → 1.0
    spread_speed = float(row.get('spread_speed', 0.0))
    velocity_score = min(spread_speed / 10.0, 1.0)

    # ── Label score (binary flag, NOT a ratio) ──────────────
    label_score = float(row.get('any_laundering', 0))

    score = (
          0.35 * fanout_score
        + 0.25 * amount_score
        + 0.25 * velocity_score
        + 0.15 * label_score
    )

    return round(min(score, 1.0), 4)


def assign_risk_tier(score: float) -> str:
    if   score >= 0.75: return 'CRITICAL'
    elif score >= 0.50: return 'HIGH'
    elif score >= 0.25: return 'MEDIUM'
    else:               return 'LOW'


print('Risk scoring functions defined ✓')

Risk scoring functions defined ✓


### Cell 2 : Apply Scoring → Produce Final `layering_df`
Scores every enriched event, assigns a tier, and sorts descending by risk
score. `layering_df` is the canonical output consumed by the GNN, dashboard,
and SAR report modules.

In [31]:
if not enriched_df.empty:

    print('Computing risk scores...')

    enriched_df['risk_score'] = enriched_df.apply(compute_layering_risk_score, axis=1)
    enriched_df['risk_tier']  = enriched_df['risk_score'].apply(assign_risk_tier)

    layering_df = enriched_df.sort_values(
        by=['risk_score', 'fanout_count'],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f'Risk scoring complete — {len(layering_df)} events scored')
    print()
    print('Risk tier distribution:')
    print(layering_df['risk_tier'].value_counts().to_string())

else:
    layering_df = enriched_df.copy()
    layering_df['risk_score'] = pd.Series(dtype=float)
    layering_df['risk_tier']  = pd.Series(dtype=str)
    print('No events to score.')

layering_df[[
    'hub_account', 'pattern_type', 'fanout_count',
    'risk_score', 'risk_tier', 'total_amount'
]].head(10)

Computing risk scores...
Risk scoring complete — 1227 events scored

Risk tier distribution:
risk_tier
MEDIUM      825
LOW         322
HIGH         65
CRITICAL     15


,hub_account,pattern_type,fanout_count,risk_score,risk_tier,total_amount
0,70_100428660,fan_out,10119,1.0,CRITICAL,1.143333e+10
1,70_1004286A8,fan_out,6117,1.0,CRITICAL,6.229317e+09
2,70_100428978,fan_out,1163,1.0,CRITICAL,1.157588e+09
3,70_1004286F0,fan_out,962,1.0,CRITICAL,2.049516e+09
4,70_100428780,fan_out,891,1.0,CRITICAL,7.733771e+10
5,70_1004288A0,fan_out,871,1.0,CRITICAL,5.699906e+08
6,70_1004289C0,fan_out,868,1.0,CRITICAL,9.534448e+08
7,70_100428810,fan_out,861,1.0,CRITICAL,5.580966e+08
8,70_100428738,fan_out,783,1.0,CRITICAL,5.444902e+10
9,70_1004287C8,fan_out,741,1.0,CRITICAL,1.174683e+10


---
## MODULE 7 — Validation & Evaluation
---

### Cell 1 : Classification Report & Confusion Matrix
Uses ground-truth `any_laundering` as the positive class. `labels=` is passed
explicitly to both `classification_report` and `confusion_matrix` so the code
does not crash if only one class is present in a given run (a common situation
on small samples).

In [32]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print('=' * 65)
print('MODULE 7 — Validation & Evaluation')
print('=' * 65)

if layering_df.empty:
    print('No events to evaluate.')

else:
    y_true = layering_df['any_laundering'].astype(int)

    DETECTION_THRESHOLD = 0.25
    y_pred = (layering_df['risk_score'] >= DETECTION_THRESHOLD).astype(int)

    print(f'\nDetection threshold : {DETECTION_THRESHOLD}')
    print(f'Total events         : {len(layering_df)}')
    print(f'Ground-truth positives (any_laundering=1) : {y_true.sum()}')
    print(f'Predicted  positives (score >= threshold) : {y_pred.sum()}')

    present_classes = sorted(y_true.unique())
    class_names     = {0: 'Clean', 1: 'Suspicious'}
    target_names    = [class_names[c] for c in present_classes]

    print('\nClassification Report:')
    print(classification_report(
        y_true, y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0
    ))

    print('Confusion Matrix (rows = actual, cols = predicted):')
    cm = confusion_matrix(y_true, y_pred, labels=present_classes)
    cm_df = pd.DataFrame(
        cm,
        index=  [f'Actual {n}' for n in target_names],
        columns=[f'Pred {n}'   for n in target_names]
    )
    print(cm_df.to_string())

    if y_true.nunique() > 1:
        auc = roc_auc_score(y_true, layering_df['risk_score'])
        print(f'\nROC-AUC Score : {auc:.4f}')
    else:
        unique_class = class_names[y_true.iloc[0]]
        print(f'\nROC-AUC : skipped — all events are "{unique_class}" (only 1 class present)')

MODULE 7 — Validation & Evaluation

Detection threshold : 0.25
Total events         : 1227
Ground-truth positives (any_laundering=1) : 393
Predicted  positives (score >= threshold) : 905

Classification Report:
              precision    recall  f1-score   support

       Clean       1.00      0.39      0.56       834
  Suspicious       0.43      1.00      0.61       393

    accuracy                           0.58      1227
   macro avg       0.72      0.69      0.58      1227
weighted avg       0.82      0.58      0.57      1227

Confusion Matrix (rows = actual, cols = predicted):
                   Pred Clean  Pred Suspicious
Actual Clean              322              512
Actual Suspicious           0              393

ROC-AUC Score : 0.8239


### Cell 2 : Per-Pattern-Type Breakdown
Shows which detector (fan-out, fan-in, rapid movement) catches the most
laundering and where detection is weakest — useful for tuning thresholds
and for deciding emphasis in the upcoming GNN training.

In [33]:
if not layering_df.empty:

    summary = layering_df.groupby('pattern_type').agg(
        total_events      = ('risk_score',     'count'),
        avg_risk_score    = ('risk_score',     'mean'),
        laundering_events = ('any_laundering', 'sum'),
        avg_total_amount  = ('total_amount',   'mean'),
        avg_fanout_count  = ('fanout_count',   'mean'),
        critical_count    = ('risk_tier', lambda x: (x == 'CRITICAL').sum()),
        high_count        = ('risk_tier', lambda x: (x == 'HIGH').sum()),
    ).reset_index()

    summary['laundering_hit_rate'] = (
        summary['laundering_events'] / summary['total_events']
    ).round(4)

    print('Per-pattern-type summary:')
    print(summary.to_string(index=False))

    print(f'\nOverall laundering hit rate : {layering_df["any_laundering"].mean():.2%}')
    print(f'Mean risk score             : {layering_df["risk_score"].mean():.4f}')
    print(f'CRITICAL tier events        : {(layering_df["risk_tier"] == "CRITICAL").sum()}')

Per-pattern-type summary:
  pattern_type  total_events  avg_risk_score  laundering_events  avg_total_amount  avg_fanout_count  critical_count  high_count  laundering_hit_rate
        fan_in           168        0.371299              140.0      1.328495e+08          5.571429               0          15               0.8333
       fan_out           234        0.400993              174.0      7.261293e+08        113.397436              15          17               0.7436
rapid_movement           825        0.312542               79.0      2.339211e+05          2.000000               0          33               0.0958

Overall laundering hit rate : 32.03%
Mean risk score             : 0.3375
CRITICAL tier events        : 15


### Cell 3 : Save Final Outputs
Persists `layering_df` and a top-50 high-risk subset to Google Drive for use
by downstream modules (structuring, dormancy, smurfing, GNN training) and
the investigator dashboard.

In [34]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/AML System/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

full_output_path  = f'{OUTPUT_DIR}/layering_detection_results.csv'
top50_output_path = f'{OUTPUT_DIR}/layering_top50_high_risk.csv'

layering_df.to_csv(full_output_path, index=False)
layering_df.head(50).to_csv(top50_output_path, index=False)

print('Outputs saved to Google Drive:')
print(f'  ✓ layering_detection_results.csv  ({len(layering_df)} rows)')
print(f'  ✓ layering_top50_high_risk.csv    (top 50 rows)')
print()
print('layering_df is ready → next: Structuring / Dormancy / Smurfing modules')
layering_df.head()

Outputs saved to Google Drive:
  ✓ layering_detection_results.csv  (1227 rows)
  ✓ layering_top50_high_risk.csv    (top 50 rows)

layering_df is ready → next: Structuring / Dormancy / Smurfing modules


,hub_account,pattern_type,counterparties,fanout_count,amounts,timestamps,is_laundering,forward_delay_seconds,fingerprint,total_amount,...,amount_std,amount_range,any_laundering,unique_counterparties,time_span_seconds,avg_tx_interval,min_tx_interval,spread_speed,risk_score,risk_tier
0,70_100428660,fan_out,"[10232_803E422C0, 10232_804AF10C0, 10232_80515...",10119,"[23317.0, 18722.79, 4815.41, 717.6, 792.92, 16...","[2022-09-01T00:01:00.000000000+00:00, 2022-09-...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_100428660::fan_out::10232_803E422C0|10232_8...,1.143333e+10,...,1.765720e+07,1.615849e+09,1.0,10119.0,86400.0,4.828164,0.0,421.625000,1.0,CRITICAL
1,70_1004286A8,fan_out,"[1024_80069C8D0, 1024_8006CED50, 1024_80070EFF...",6117,"[834.69, 3673.87, 18741.89, 146614.28, 1962.5,...","[2022-09-01T00:12:00.000000000+00:00, 2022-09-...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_1004286A8::fan_out::1024_80069C8D0|1024_800...,6.229317e+09,...,1.306488e+07,8.258079e+08,1.0,6117.0,86400.0,8.170213,0.0,254.875000,1.0,CRITICAL
2,70_100428978,fan_out,"[112_8000C7240, 112_80DBE80A0, 112_80DBED0B0, ...",1163,"[5474.84, 56042.84, 1542.07, 7142.88, 1803957....","[2022-09-01T00:20:00.000000000+00:00, 2022-09-...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_100428978::fan_out::112_8000C7240|112_80DBE...,1.157588e+09,...,1.372335e+07,5.960868e+08,1.0,1163.0,86400.0,42.793462,0.0,48.458333,1.0,CRITICAL
3,70_1004286F0,fan_out,"[112078_8058B6600, 112078_806F0C3D0, 112078_80...",962,"[26780.58, 154765.34, 508404.39, 7138.41, 3506...","[2022-09-01T05:20:00.000000000+00:00, 2022-09-...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_1004286F0::fan_out::112078_8058B6600|112078...,2.049516e+09,...,1.088700e+07,2.866118e+08,1.0,962.0,86400.0,44.467319,0.0,40.083333,1.0,CRITICAL
4,70_100428780,fan_out,"[112064_8048204B0, 112064_804824390, 112064_80...",891,"[152.58, 1542.58, 91915.07, 336528.23, 108101....","[2022-09-01T03:07:00.000000000+00:00, 2022-09-...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",NaN,70_100428780::fan_out::112064_8048204B0|112064...,7.733771e+10,...,7.384377e+08,2.716420e+10,1.0,891.0,86340.0,50.373396,0.0,37.150799,1.0,CRITICAL
